## Testing Padding = 0 instead of default Padding = 1 


Use the same seed 7 to create the training split. Then use seed.126 to control the setup of the starting weights so that i can directy compare to the padding - 1 version.  

In [4]:
from pathlib import Path
from time import perf_counter

import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor

# Find data/train from the repo root or the Alice folder.
train_path = Path("data/train")
if not train_path.is_dir():
    train_path = Path("../../data/train")

train_data = ImageFolder(train_path, transform=ToTensor())

train_indices, val_indices = train_test_split(
    range(len(train_data)),
    test_size=0.20,
    random_state=7,
    stratify=train_data.targets,
)

tuning_train = Subset(train_data, train_indices)
validation = Subset(train_data, val_indices)

print(f"Training images: {len(tuning_train):,}")
print(f"Validation images: {len(validation):,}")

Training images: 17,280
Validation images: 4,320


In [5]:
seed = 126
num_epochs = 10

torch.manual_seed(seed)

padding_0_model = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(16, 32, kernel_size=3, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(32 * 14 * 14, 10),
)

optimizer = torch.optim.Adam(padding_0_model.parameters(), lr=0.001)
loss_function = nn.CrossEntropyLoss()

train_loader = DataLoader(
    tuning_train,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
val_loader = DataLoader(validation, batch_size=32, shuffle=False)

padding_0_history = []

for epoch in range(1, num_epochs + 1):
    padding_0_model.train()
    loss_sum = 0.0
    start_time = perf_counter()

    for images, labels in train_loader:
        optimizer.zero_grad()
        scores = padding_0_model(images)
        loss = loss_function(scores, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * labels.size(0)

    training_seconds = perf_counter() - start_time
    training_loss = loss_sum / len(tuning_train)

    padding_0_model.eval()
    correct = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            predictions = padding_0_model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()

    validation_accuracy = correct / len(validation)
    padding_0_history.append({
        "epoch": epoch,
        "training_loss": training_loss,
        "validation_accuracy": validation_accuracy,
        "training_seconds": training_seconds,
    })

    print(
        f"Epoch {epoch:2d}/{num_epochs} | "
        f"Loss {training_loss:.3f} | "
        f"Validation accuracy {validation_accuracy:.2%} | "
        f"Training time {training_seconds:.1f}s"
    )

best = max(padding_0_history, key=lambda row: row["validation_accuracy"])
total_time = sum(row["training_seconds"] for row in padding_0_history)

print("\nPadding 0 summary")
print(f"Epoch 10 validation accuracy: {padding_0_history[-1]['validation_accuracy']:.2%}")
print(f"Best validation accuracy: {best['validation_accuracy']:.2%} at epoch {best['epoch']}")
print(f"Total training time, excluding validation: {total_time:.1f}s")

Epoch  1/10 | Loss 1.432 | Validation accuracy 63.84% | Training time 16.3s
Epoch  2/10 | Loss 0.956 | Validation accuracy 69.51% | Training time 19.3s
Epoch  3/10 | Loss 0.826 | Validation accuracy 73.15% | Training time 18.7s
Epoch  4/10 | Loss 0.749 | Validation accuracy 71.16% | Training time 18.6s
Epoch  5/10 | Loss 0.699 | Validation accuracy 76.50% | Training time 19.2s
Epoch  6/10 | Loss 0.663 | Validation accuracy 76.18% | Training time 19.0s
Epoch  7/10 | Loss 0.611 | Validation accuracy 79.49% | Training time 18.8s
Epoch  8/10 | Loss 0.600 | Validation accuracy 78.12% | Training time 19.1s
Epoch  9/10 | Loss 0.548 | Validation accuracy 79.65% | Training time 18.8s
Epoch 10/10 | Loss 0.547 | Validation accuracy 79.86% | Training time 18.9s

Padding 0 summary
Epoch 10 validation accuracy: 79.86%
Best validation accuracy: 79.86% at epoch 10
Total training time, excluding validation: 186.7s


### Results 

Can see that Epoch 10 validation accuracy was 80.86% with Padding=1 and 79.86% with Padding=0 so in terms of accuracy using Padding=1 seems to be better. 
However it is interesting to see that the taining time was 23.9 seconds quicker for the Padding=0 but this is quite small amount. 
But percentage wise it is 13.74% quicker so maybe if you were scaling this up to many more image I would look at using the padding=0 and trading off a small amount of accuracy.